In [1]:
import os
os.chdir("/home/f/GraOmicSynergy/")

In [2]:
import gc
import os
import pickle
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, ReLU, Sequential
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from tqdm.auto import tqdm

from utils import *

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

AMP_DTYPE = torch.float16


In [ ]:
def return_omics_type(data):
    active = [name for name, enabled in data.items() if enabled]
    return f"{len(active)}omics", "_".join(active)

data_types = [
    {"ge": 1, "mut": 0, "meth": 0},
    {"ge": 0, "mut": 1, "meth": 0},
    {"ge": 0, "mut": 0, "meth": 1},
    {"ge": 1, "mut": 1, "meth": 0},
    {"ge": 1, "mut": 0, "meth": 1},
    {"ge": 0, "mut": 1, "meth": 1},
    {"ge": 1, "mut": 1, "meth": 1},
]
data_sets = ["all_test"]
early_stopping_patience = 20
min_delta = 1e-4

TRAIN_CONFIG = {
    "lr": 0.001,
    "weight_decay": 1e-4,
    "num_epoch": 300,
    "batch_size": 1024,
}

EVAL_SCENARIOS = [
    {"metric_name": "test", "val_dataset": "val_dc", "test_dataset": "test_dc", "detail_name": "test", "fig_suffix": ""},
    {"metric_name": "mix_dc", "val_dataset": "mix_val", "test_dataset": "mix_test", "detail_name": "mix", "fig_suffix": "_mix_dc"},
    {"metric_name": "blind_cell", "val_dataset": "blind_cell_val", "test_dataset": "blind_cell_test", "detail_name": "blind_cell", "fig_suffix": "_blind_cell"},
    {"metric_name": "blind_1_drug", "val_dataset": "blind_1_drug_val", "test_dataset": "blind_1_drug_test", "detail_name": "blind_1_drug", "fig_suffix": "_blind_1_drug"},
    {"metric_name": "blind_1_drug_cell", "val_dataset": "blind_1_drug_cell_val", "test_dataset": "blind_1_drug_cell_test", "detail_name": "blind_1_drug_cell", "fig_suffix": "_blind_1_drug_cell"},
    {"metric_name": "blind_2_drug", "val_dataset": "blind_2_drug_val", "test_dataset": "blind_2_drug_test", "detail_name": "blind_2_drug", "fig_suffix": "_blind_2_drug"},
    {"metric_name": "blind_all", "val_dataset": "blind_all_val", "test_dataset": "blind_all_test", "detail_name": "blind_all", "fig_suffix": "_blind_all"},
]

SHM_ROOT = Path("/dev/shm/GraOmicSynergy")

# KHẮC PHỤC CỐT LÕI: Thêm x_f1 và x_f2 vào follow_batch
LOADER_KWARGS = dict(follow_batch=["x_1", "x_2", "x_f1", "x_f2"], pin_memory=True, persistent_workers=True, prefetch_factor=2)

TRAIN_LOADER_KWARGS = dict(LOADER_KWARGS, num_workers=4)
EVAL_LOADER_KWARGS = dict(LOADER_KWARGS, num_workers=1)
PROGRESS_KWARGS = dict(mininterval=10, maxinterval=30, miniters=1, leave=False, dynamic_ncols=False)

def processed_splits():
    splits = {"train_dc"}
    for scenario in EVAL_SCENARIOS:
        splits.update((scenario["val_dataset"], scenario["test_dataset"]))
    return sorted(splits)

def prepare_shm_dataset(data_set, dataset_prefix="GDSC"):
    disk_dir = Path("data/split_data") / data_set / "processed"
    shm_dir = SHM_ROOT / "split_data" / data_set / "processed"
    shm_dir.mkdir(parents=True, exist_ok=True)
    for split in processed_splits():
        src = disk_dir / f"{dataset_prefix}_{split}.pt"
        dst = shm_dir / src.name
        if not dst.exists() or src.stat().st_size != dst.stat().st_size or src.stat().st_mtime_ns != dst.stat().st_mtime_ns:
            shutil.copy2(src, dst)
    return str(shm_dir.parent)

def make_loader(root, split, batch_size, shuffle, loader_kwargs, dataset_prefix="GDSC"):
    dataset = TestbedDataset(root=root, dataset=f"{dataset_prefix}_{split}")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, **loader_kwargs)

In [ ]:
class GINEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_layers=5):
        super(GINEncoder, self).__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
        return x

class MultiOmicsCNN(nn.Module):
    def __init__(self, out_dim=128, ge=1, mut=1, meth=1):
        super().__init__()
        self.ge, self.mut, self.meth = ge, mut, meth
        def cnn_block():
            return nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.ReLU(),
                nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
                nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
                nn.AdaptiveMaxPool1d(1), nn.Flatten()
            )
        if self.ge: self.ge_conv = cnn_block()
        if self.mut: self.mut_conv = cnn_block()
        if self.meth: self.meth_conv = cnn_block()
        
        total_omics = self.ge + self.mut + self.meth
        self.fusion = nn.Linear(128 * total_omics, out_dim)

    def forward(self, ge, mut, meth):
        feats = []
        if self.ge: feats.append(self.ge_conv(ge.unsqueeze(1)))
        if self.mut: feats.append(self.mut_conv(mut.unsqueeze(1)))
        if self.meth: feats.append(self.meth_conv(meth.unsqueeze(1)))
        return self.fusion(torch.cat(feats, dim=1))

class FragmentGINNet(nn.Module):
    def __init__(self, num_features_atom=78, num_features_frag=2300, hidden_dim=128, ge=1, mut=1, meth=1):
        super().__init__()
        # MODULE 1: Nhánh Dòng Tế Bào
        self.cell_encoder = MultiOmicsCNN(out_dim=hidden_dim, ge=ge, mut=mut, meth=meth)
        
        # MODULE 2: Nhánh Thuốc (Dual-View)
        self.atom_encoder = GINEncoder(num_features_atom, hidden_dim, num_layers=5)
        self.frag_encoder = GINEncoder(num_features_frag, hidden_dim, num_layers=2)
        self.alpha_param = nn.Parameter(torch.tensor(0.0)) # Cổng Sigmoid Gate
        
        # MODULE 3: Dung Hợp Cặp Thuốc (Attention Giao Hoán)
        self.attn_linear = nn.Linear(hidden_dim * 3, 1)
        
        # MODULE 4: Khối Dự Đoán
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

    def forward(self, data):
        # 1. Xử lý đa omics dòng tế bào
        cn = self.cell_encoder(data.target_ge, data.target_mut, data.target_meth)
        
        # 2. Xử lý góc nhìn kép nội bộ thuốc (Intra-Drug Fusion)
        alpha = torch.sigmoid(self.alpha_param)
        
        # Thuốc A
        za_A = global_add_pool(self.atom_encoder(data.x_1, data.edge_index_1), data.x_1_batch)
        zf_A = global_add_pool(self.frag_encoder(data.x_f1, data.edge_index_f1), data.x_f1_batch)
        dA = alpha * za_A + (1 - alpha) * zf_A
        
        # Thuốc B
        za_B = global_add_pool(self.atom_encoder(data.x_2, data.edge_index_2), data.x_2_batch)
        zf_B = global_add_pool(self.frag_encoder(data.x_f2, data.edge_index_f2), data.x_f2_batch)
        dB = alpha * za_B + (1 - alpha) * zf_B
        
        # 3. Cơ chế chú ý cặp thuốc (Inter-Drug Fusion)
        eA = torch.exp(F.leaky_relu(self.attn_linear(torch.cat([dA, cn, dB], dim=1))))
        eB = torch.exp(F.leaky_relu(self.attn_linear(torch.cat([dB, cn, dA], dim=1))))
        
        attn_A = eA / (eA + eB + 1e-8)
        attn_B = eB / (eA + eB + 1e-8)
        dpair = attn_A * dA + attn_B * dB
        
        # 4. Dự đoán đầu ra
        output = self.predictor(torch.cat([dpair, cn], dim=1))
        return output, attn_A, attn_B

In [ ]:
def train(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = torch.zeros((), device=device)
    for data in loader:
        data = data.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
            output, _, _ = model(data)
            loss = criterion(output, data.y.view(-1, 1).float())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.detach()
    return (total_loss / len(loader)).item()

In [ ]:
def predict(model, loader, return_attention=False):
    model.eval()
    labels, preds, drug_1_att, drug_2_att = [], [], [], []

    with torch.inference_mode():
        for data in loader:
            data = data.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
                output, a_drug_1, a_drug_2 = model(data)
            labels.append(data.y.view(-1, 1).float().cpu())
            preds.append(output.float().cpu())
            drug_1_att.append(a_drug_1.float().cpu())
            drug_2_att.append(a_drug_2.float().cpu())

    labels = torch.cat(labels).numpy().ravel()
    preds = torch.cat(preds).numpy().ravel()
    if return_attention:
        return labels, preds, torch.cat(drug_1_att).numpy().ravel(), torch.cat(drug_2_att).numpy().ravel()
    return labels, preds

In [ ]:
def return_ret(G, P):
    return [rmse(G,P),mse(G,P),pearson(G,P),spearman(G,P)]

def get_top_data(r, df):
    G, P, x_1_ats, x_2_ats = r
    x_1_ats = np.asarray(x_1_ats)
    x_2_ats = np.asarray(x_2_ats)
    abs_error = np.abs(G-P)
    index_top = abs_error.argsort()
    df_top = df.iloc[index_top].copy()
    df_top["log_synergy"] = G[index_top]
    df_top["predict"] = P[index_top]
    df_top["abs_error"] = abs_error[index_top]
    df_top["x_1_ats"] = x_1_ats[index_top]
    df_top["x_2_ats"] = x_2_ats[index_top]
    return df_top

# Train model

In [ ]:
from mol_bpe import Tokenizer
tokenizer = Tokenizer("vocab.txt")
FRAG_VOCAB_SIZE = tokenizer.num_subgraph_type()

MODEL_ST = "FragmentGINNet"
DATASET_PREFIX = "GDSC"
runs = [(data_type, data_set) for data_type in data_types for data_set in data_sets]
print("Omics runs:", ", ".join(return_omics_type(data_type)[1] for data_type in data_types))
print("Eval scenarios:", ", ".join(scenario["metric_name"] for scenario in EVAL_SCENARIOS))

for run_idx, (data_type, data_set) in enumerate(tqdm(runs, desc="omics runs", **PROGRESS_KWARGS), 1):
    cfg = TRAIN_CONFIG
    batch_size = cfg["batch_size"]
    data_path = prepare_shm_dataset(data_set, DATASET_PREFIX)
    split_path = f"data/split_data/{data_set}/"
    omics, name_omics = return_omics_type(data_type)

    print("\nrunning on", f"{MODEL_ST}_{DATASET_PREFIX}", f"{run_idx}/{len(runs)}", name_omics)
    print(device)
    print("Training config:", cfg)

    train_loader = make_loader(data_path, "train_dc", batch_size, True, TRAIN_LOADER_KWARGS, DATASET_PREFIX)
    loaders = {
        scenario["metric_name"]: (
            make_loader(data_path, scenario["val_dataset"], batch_size, False, EVAL_LOADER_KWARGS, DATASET_PREFIX),
            make_loader(data_path, scenario["test_dataset"], batch_size, False, EVAL_LOADER_KWARGS, DATASET_PREFIX),
            scenario,
        )
        for scenario in EVAL_SCENARIOS
    }

    # Khởi tạo mô hình cấu trúc FragmentGINNet mới
    model = FragmentGINNet(num_features_atom=78, num_features_frag=FRAG_VOCAB_SIZE, hidden_dim=128, **data_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = nn.SmoothL1Loss() # Giữ nguyên SmoothL1 Loss như yêu cầu
    scaler = torch.amp.GradScaler("cuda")

    save_path = f"models/save_model_new/ALL_TEST_GIN_ADD_AT/{omics}/{name_omics}/{data_set}/"
    print(save_path)

    model_file_name = save_path + "best_model.model"
    result_file_name = save_path + f"result_{MODEL_ST}_{DATASET_PREFIX}.csv"
    loss_fig_name = save_path + f"model_{MODEL_ST}_{DATASET_PREFIX}_loss"
    pearson_fig_name = save_path + f"model_{MODEL_ST}_{DATASET_PREFIX}_pearson"

    scenario_names = [scenario["metric_name"] for scenario in EVAL_SCENARIOS]
    train_losses = []
    val_losses = {name: [] for name in scenario_names}
    val_pearsons = {name: [] for name in scenario_names}
    best_val_losses = {name: float("inf") for name in scenario_names}
    ret_test_save = {name: [1, 1, 1, 1] for name in scenario_names}
    best_states = {}
    bad_epochs = {name: 0 for name in scenario_names}
    active = scenario_names.copy()

    run_started = time.time()
    epoch_bar = tqdm(range(1, cfg["num_epoch"] + 1), desc=f"{run_idx}/{len(runs)} {name_omics}", **PROGRESS_KWARGS)
    epoch_bar.set_postfix()
    for epoch in epoch_bar:
        train_loss = train(model, train_loader, optimizer, loss_fn, scaler)
        train_losses.append(train_loss)

        for name in active.copy():
            val_loader, test_loader, _ = loaders[name]
            val_ret = return_ret(*predict(model, val_loader, return_attention=False))
            val_losses[name].append(val_ret[1])
            val_pearsons[name].append(val_ret[2])

            if val_ret[1] < best_val_losses[name] - min_delta:
                best_val_losses[name] = val_ret[1]
                bad_epochs[name] = 0
                ret_test_save[name] = return_ret(*predict(model, test_loader, return_attention=False))
                best_states[name] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                continue

            bad_epochs[name] += 1
            if bad_epochs[name] >= early_stopping_patience:
                active.remove(name)
                
        if not active:
            break

    os.makedirs(save_path, exist_ok=True)
    model.load_state_dict(best_states["test"])
    torch.save(best_states["test"], model_file_name)

    with open(result_file_name, "w") as f:
        for scenario in EVAL_SCENARIOS:
            name = scenario["metric_name"]
            f.write("ret_" + name + ": " + ",".join(map(str, ret_test_save[name])) + "\n")

    for scenario in EVAL_SCENARIOS:
        name = scenario["metric_name"]
        draw_loss(train_losses[:len(val_losses[name])], val_losses[name], loss_fig_name + scenario["fig_suffix"])
        draw_pearson(val_pearsons[name], pearson_fig_name + scenario["fig_suffix"])

    log = [train_losses]
    log.extend(val_losses[name] for name in scenario_names)
    log.extend(val_pearsons[name] for name in scenario_names)
    with open(save_path + "log.pickle", "wb") as f:
        pickle.dump(log, f)

    for name, (_, test_loader, scenario) in tqdm(loaders.items(), desc="details", **PROGRESS_KWARGS):
        model.load_state_dict(best_states[name])
        result = get_top_data(
                predict(model, test_loader, return_attention=True),
                pd.read_csv(split_path + scenario["test_dataset"] + ".csv"),
            )
        result.to_csv(save_path + f"detail_result_{scenario['detail_name']}.csv", index=False)

    del model, train_loader, loaders, best_states
    torch.cuda.empty_cache()
    gc.collect()

Omics runs: ge, mut, meth, ge_mut, ge_meth, mut_meth, ge_mut_meth
Eval scenarios: test, mix_dc, blind_cell, blind_1_drug, blind_1_drug_cell, blind_2_drug, blind_all


omics runs:   0%|          | 0/7 [00:00<?, ?it/s]


running on GINConvNet_GDSC 1/7 ge
cuda
Training config: {'lr': 0.001, 'weight_decay': 0.0001, 'num_epoch': 300, 'batch_size': 1024}
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_train_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_val_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_test_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_mix_val.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_mix_test.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_blind_cell_val.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_blind_cell_test.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_bli

1/7 ge:   0%|          | 0/300 [00:00<?, ?it/s]

details:   0%|          | 0/7 [00:00<?, ?it/s]


running on GINConvNet_GDSC 2/7 mut
cuda
Training config: {'lr': 0.001, 'weight_decay': 0.0001, 'num_epoch': 300, 'batch_size': 1024}
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_train_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_val_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_test_dc.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_mix_val.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_mix_test.pt, loading ...
Pre-processed data found: /dev/shm/GraOmicSynergy/split_data/all_test/processed/GDSC_blind_cell_val.pt, loading ...
